# Submission — &lt;Parmesan&gt;

Copy this notebook into `submissions/<team-handle>/submission.ipynb` and
fill in every section. Judges run this notebook top-to-bottom on a clean
machine — make sure it executes.

> AI Usage Disclaimer: We do use AI for code generation. But all documentation in this JupyterNotebook is written by us, no AI is used for any markdown cells; 

## 1 · Setup

In [ ]:
import os

import numpy as np
import uniqx

GATEWAY = os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443")
uniqx.login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY)
client = uniqx.connect(GATEWAY)
print("uniqx", uniqx.__version__)

# Resolve the target directory relative to the notebook's current working directory
import sys
CFD_PATH = os.path.abspath("../../../tracks/cfd")
if CFD_PATH not in sys.path:
    sys.path.insert(0, CFD_PATH)
    
print("CFD path:", CFD_PATH)

## 2 · Workload

In this task, we solving the 2D incompressible Navier-Stokes equation with Chorin's projection method.


$$
\frac{\partial \mathbf{u}}{\partial t} + \left(\mathbf{u} \cdot \nabla\right) \mathbf{u} = - \frac{1}{\rho} \nabla p + \nu \nabla^2 \mathbf{u}
$$

we consider Dirichlet B.C. as specified under `boundary.py` script:

<p align="center">
  <img src="assets/2d_domain.png" alt="2D Domain with Dirichlet Boundary Conditions" width="400"/><br>
  <em>Dirichlet B.C. with top-wall has net inflow in x-direction $u = U_{\text{lid}}$</em>
</p>

Note that with the choice of $ U_{\text{lid}} = 1 m / s$ and  , as the task specified in `cfd/config.py`, the Reynolds number can be computed as

$$
\text{Re} = \frac{U_{\text{lid}} L}{\nu} = 100
$$


We want to point out that, in the `cfd/config.py` file, the provided comment mistakenly states with a Reynold number of $100$, the flow dynamics falls into the "Stokes regime". 

```python
# --- Boundary condition ---
U_LID = 1.0     # lid (top wall) velocity in x-direction [m/s]
                # => Reynolds number Re = U_LID * L / NU = 100 (Stokes regime)
```


Note that although this is a low Reynold number, but it is not low enough to entirely fall into that regime,  The Stokes flow should demonstrate $\text{Re} \ll 1$, as commonly used in standard textbooks. Nevertheless, we agree on referring to it as a "Stokes-like" system,  with viscous forces being more dominant, and consequently, renders the solution with the projection method suitable. We will introduce in the methodology section how this "Stokes-like" behavior allows the decoupled computation of velocity and pressure.



### Methodology: Chorin's Projection Method


<u>Motivation: Helmholtz decomposition</u>

<p align="center">
<img src="assets/helmholtz.png" alt="Helmholtz Decomposition Diagram" width="500">
</p>

<p align="center"><em>Image Source: <a href="https://www.linkedin.com/posts/alechelbling_the-helmholtz-decomposition-is-one-of-the-ugcPost-7462282011861356544-anpu?utm_source=share&utm_medium=member_desktop&rcm=ACoAADCyUoYByXv17KranNM32Csw2ZBPghg_ZU0">By Alec Helbing</a></em></p>



<u>Derivation of Poisson Equation</u>


In the following, denote "sol - solenoidal",

$$
\mathbf{u} = \mathbf{u}_{\text{sol}} + \mathbf{u}_{\text{irrot}} = \mathbf{u}_{\text{sol}} + \nabla \phi
$$



By taking the divergence on the equation, we arrive at the *Poisson equation* for the scalar function $\phi$

$$
 \begin{align*}
 \nabla \cdot \mathbf{u} &= \nabla \cdot \left(\mathbf{u}_{\text{sol}} + \nabla \phi\right) \\
 &\Leftrightarrow \nabla \cdot \mathbf{u} = \cancel{\nabla \cdot \mathbf{u}_{\text{sol}}} + \nabla \cdot \left(\nabla \phi\right) \\
&\Leftrightarrow \nabla \cdot \mathbf{u} = \nabla^2 \phi
 \end{align*}
$$


We see that for known vector field $\mathbf{u}$, we can solve for the scalar-valued $\phi$; in fact, we consider pressure as the underlying quantity in our application, in which let $\phi := P$ and have the form we see in <u>B - Pressure Poisson</u> with some additional constants $\rho, \Delta t$ for the density, discrete time step size accordingly.



$$
\nabla^2 P = \frac{\rho}{\Delta t} \nabla \cdot \mathbf{u}^{\star}
$$



In [ ]:
# REPLACE — your traced module construction.
# Example pattern:
#   import uniqx
#   @uniqx.to_module(name="my_kernel")
#   def my_kernel(x):
#       return uniqx.ops.matmul(x, x)
#   module = my_kernel
#   runtime_inputs = [your_runtime_data]
module = None
runtime_inputs = []
assert module is not None, "build your traced module above"

## 3 · Preflight — the Pareto table

Required: print `options.summary()` and copy the output into
`preflight_log.txt`.

In [ ]:
options = uniqx.preflight(module, client=client)
print(options.summary())
try:
    options.plot()
except Exception as exc:
    print(f"plot skipped: {exc}")

## 4 · Submit

State which option you picked and *why* in the markdown cell below the
submission. The `why` paragraph goes verbatim into `results.json.justification`.

In [ ]:
choice = options.recommended    # or pick a non-recommended row
print(f"Selected: {choice['label']}")

job_id = uniqx.submit(
    module,
    client=client,
    preflight_job_id=options.job_id,
    option_idx=choice["_idx"],
    runtime_inputs=runtime_inputs,
)
result = uniqx.get(job_id, client=client, timeout=600)
print("job complete")

**Why this option:** REPLACE — 3-6 sentences referencing specific numbers
from the preflight table above.

## 5 · Baseline comparison

### 5.1 · Understanding Baseline Solver(s)

We may take a small peek into the baseline and see how it is implemented, in particular, to understand why the JAX library is needed. In `linalg.py`, we see the JAX solver backends implement two solvers:

1. Direct solver (`solve_direct`) 
    - dense LU `dense LU via jnp.linalg.solve`

2. Iterative solver (`solve_cg`)
    - conjugate gradient `jax.scipy.sparse.linalg.cg`

Testing out what configuration to choose, we should quickly notice the iterative solver is unstable. Indeed, the system is ill-conditioned, showing difficulties to reach better accuracy, and therefore fails to meet the tolerance requirement and forces the solver to results in more iteration steps.

The standard approach, in combination with a conjugate gradient solver, is to apply a preconditioner on the target LSE to resolve. But to not modify the baseline (in fact, we do this in the improved solver, by setting `hermitian=True`, we essentially tell the solver the system is SPD, and is suitable for a Cholesky factorization). Here, for a fair baseline, we alternatively select a less restricted tolerance requirement of $1e^{-4}$ than that of the direct solver.    

### 5.2 · Configuration Setup

The choice of grid size variation is made for `N=8, 16, 32`. For audience that are familiar with classical numerical simulations, this choice of spatial resolution seems like a toy example. But our choice is made under the consideration of the backend support.



In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from IPython.display import Image, display

# Notebook CWD is its own directory when launched via Jupyter.
ASSETS_DIR   = Path("assets")
BASELINE_DIR = ASSETS_DIR / "20260522_025600_N8-16-32_direct-cg"

with open(BASELINE_DIR / "jax_benchmark.json") as fh:
    bench = json.load(fh)

print(f"Loaded benchmark  : grid_sizes={bench['grid_sizes']}, bench_n={bench['bench_n']}")
print(f"Backends in totals: {sorted({e['backend'] for e in bench['totals']})}")

In [ ]:
# ── Chart 1: Throughput (ms / step) by backend × grid size ────────────────────
grid_sizes = bench["grid_sizes"]       # [8, 16, 32]
backends   = ["jax/direct", "jax/cg"]

perf = {(e["backend"], e["N"]): e["ms_per_step"] for e in bench["totals"]}

x     = np.arange(len(grid_sizes))
width = 0.25
colors = {"jax/direct": "#4C72B0", "jax/cg": "#DD8452"}

fig, ax = plt.subplots(figsize=(8, 5))
for i, backend in enumerate(backends):
    vals = [perf[(backend, N)] for N in grid_sizes]
    ax.bar(x + i * width, vals, width, label=backend, color=colors[backend])

# TODO: replace nans with actual uniqx/compiled timing once gateway run is complete
ax.bar(x + 2 * width, [float("nan")] * len(grid_sizes), width,
       label="uniqx/compiled (TODO)", color="#55A868", alpha=0.4,
       hatch="//", edgecolor="grey")

ax.set_xticks(x + width)
ax.set_xticklabels([f"N={N}" for N in grid_sizes])
ax.set_ylabel("ms / step")
ax.set_title("Throughput: JAX baseline vs. uniqx (Challenge 1)")
ax.legend()
ax.set_yscale("log")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}"))
plt.tight_layout()
plt.savefig(ASSETS_DIR / "throughput_comparison.png", dpi=150)
plt.show()
print("Saved → assets/throughput_comparison.png")

In [ ]:
# ── Chart 2: Per-stage breakdown at N=32 (stacked bars) ───────────────────────
stages  = bench["stages"]    # keys: "direct", "cg"
solvers = ["direct", "cg"]
stage_keys = [
    ("A_ms", "Stage A — Diffusion stencil",  "#4C72B0"),
    ("B_ms", "Stage B — Poisson solve",      "#DD8452"),
    ("C_ms", "Stage C — Correction stencil", "#55A868"),
]

x       = np.arange(len(solvers))
bar_w   = 0.4
fig, ax = plt.subplots(figsize=(7, 5))
bottoms = np.zeros(len(solvers))

for key, label, color in stage_keys:
    vals   = np.array([stages[s][key] for s in solvers])
    totals = np.array([stages[s]["total_ms"] for s in solvers])
    ax.bar(x, vals, bar_w, bottom=bottoms, label=label, color=color)
    for xi, (v, b, tot) in enumerate(zip(vals, bottoms, totals)):
        if v / tot > 0.04:
            ax.text(xi, b + v / 2, f"{v/tot*100:.0f}%",
                    ha="center", va="center", fontsize=10, color="white", fontweight="bold")
    bottoms += vals

ax.set_xticks(x)
ax.set_xticklabels([f"jax/{s}" for s in solvers])
ax.set_ylabel(f"ms / step  (harmonic mean, N={bench['bench_n']})")
ax.set_title(f"Per-stage cost at N={bench['bench_n']}  —  Stage B dominates")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(ASSETS_DIR / "stage_breakdown.png", dpi=150)
plt.show()

for s in solvers:
    st = stages[s]
    print(f"jax/{s}:  A={st['A_ms']:.2f} ms  B={st['B_ms']:.2f} ms  "
          f"C={st['C_ms']:.2f} ms  (B={st['B_ms']/st['total_ms']*100:.0f}% of total)")
print("Saved → assets/stage_breakdown.png")

In [ ]:
# ── Convergence figures (500 steps, N=32) ─────────────────────────────────────
for solver in ["direct", "cg"]:
    fig_path = BASELINE_DIR / f"results_jax_{solver}.png"
    print(f"--- jax/{solver} ---")
    display(Image(filename=str(fig_path), width=700))

In [ ]:
# ── Baseline metrics summary ───────────────────────────────────────────────────
# JAX/direct at N=32 is the primary reference for the uniqx/compiled comparison.
perf_lookup   = {(e["backend"], e["N"]): e["ms_per_step"] for e in bench["totals"]}
jax_direct_ms = perf_lookup[("jax/direct", 32)]
jax_direct_s  = jax_direct_ms * bench["timing_steps"] / 1e3

# TODO: replace with actual uniqx/compiled timing once gateway run is complete
runtime_s          = jax_direct_s   # placeholder — JAX/direct baseline
accuracy_rel_error = 0.0            # TODO: compute from velocity-field L2 comparison

print(f"JAX/direct N=32    : {jax_direct_ms:.2f} ms/step  ({jax_direct_s:.2f} s for {bench['timing_steps']} steps)")
print(f"runtime_s          : {runtime_s:.4f}  (placeholder — update with uniqx result)")
print(f"accuracy_rel_error : {accuracy_rel_error}  (TODO)")

### TODO — items required by the README not yet completed

| # | Item | Needed for |
|---|------|-----------|
| 1 | Run `main.py` via gateway; record **uniqx/compiled ms/step at N=8, 16, 32** | Chart 1 third bar; `results.json.metrics.runtime_s` |
| 2 | Compute **`accuracy_rel_error`** — L2 norm of velocity-field diff between JAX/direct and uniqx/compiled at N=32 | `results.json.metrics.accuracy_rel_error`; Performance score ≥ 25 pts |
| 3 | Fill **`results.json.metrics.cost_usd`** and **`carbon_g`** from the gateway preflight table | Tie-breaker; Tradeoff reasoning score |
| 4 | Re-run `scripts/run_jax_benchmark.sh` to recreate the missing **`assets/latest` symlink** | Notebook path robustness on clean machine |
| 5 | Update placeholder `runtime_s = 0.0` in `results.json` with actual uniqx result | `results.json` validity; Robustness score |
| 6 | Write Section 6 discussion prose (3–6 sentences) referencing specific numbers from Charts 1 & 2 | Tradeoff reasoning score ≥ 18 pts |

## 6 · Discussion

REPLACE — short prose answering:

1. What did the Pareto frontier look like?
2. What did you learn from comparing baselines?
3. Where would you push next with more time?